# Importing

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Importing                             #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! source /home/ivm/envs/valid_env/bin/activate
import polars as pl
import sys
sys.path.append(("/home/ivm/valid/scripts/utils/"))
from general_utils import *
from model_eval_utils import compare_models

# For shap
from plot_utils import get_plot_names
import pickle
import shap

%load_ext autoreload
%autoreload 2

# Settings

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Settings                              #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
base_path="/home/ivm/valid/data/processed_data/pred_2026-01"
%env base_path=$base_path
%env fg_ver=R13
%env lab_name=egfr
%env extra=d1_KDIGO-soft2_ld
%env diff=90
%env lab_name_three=uacr
%env extra_three=d1_ld
%env date_three=2025-11-05
%env lab_name_two=cystc
%env extra_two=d1_KDIGO-strict_ld
%env date_two=2025-11-05
%env extra_labels=testv10_2022_w3
%env date_1=2025-11-05
%env date_2=2026-01-07
%env date_3=2026-01-07
%env date_4=2026-01-07
%env date_5=2026-01-07
base_date = datetime(2021,10,1)
%env base_date=2021-10-01
%env start_pred_date=2022-01-01
%env end_pred_date=2022-12-31
%env min_age=30
%env max_age=70
%env months_buffer=3
%env abnorm_type=strong
%env valid_pct=0.3
%env test_pct=0.0
%env version=v10
%env bin_goal=y_MEAN_ABNORM
%env cont_goal=y_MEAN

goal="y_MEAN_ABNORM"
file_descr = "testv10_2022_w3"
lab_name = "egfr"
base_date = datetime(2021,10,1)


# Filters

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                 Filters                                                 #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
data = pl.read_parquet(base_path+"/step1_clean/egfr_d1_KDIGO-soft2_ld_2025-11-05.parquet")

train_goals = {"mglogloss": "y_MEAN_ABNORM", "logloss": "y_MEAN_ABNORM",  "q90": "y_MEAN", "q95":"y_MEAN", "q75": "y_MEAN", "q25": "y_MEAN", "q10": "y_MEAN", "mae": "y_MEAN"}

lastval_long_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.DATE.dt.year()<2020)["FINNGENID"]))
no_history_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"]))
history_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"]))
last_norm_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.ABNORM_CUSTOM<1)["FINNGENID"]))
no_abnorm_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter(pl.col.ABNORM_CUSTOM==1)["FINNGENID"]))
thirty_filter = (((pl.col.EVENT_AGE>=30)&(pl.col.EVENT_AGE<40)))
fourty_filter = (((pl.col.EVENT_AGE>=40)&(pl.col.EVENT_AGE<50)))
fifty_filter = (((pl.col.EVENT_AGE>=50)&(pl.col.EVENT_AGE<60)))
sixty_filter = (((pl.col.EVENT_AGE>=60)&(pl.col.EVENT_AGE<=70)))

set_names = {0: "Train", 1: "Valid", 2: "Test"}
test_filter = pl.col.SET==2
valid_filer = pl.col.SET==1
train_filter = pl.col.SET==0

filters = {"All": True, 
           "History": history_filter, 
           "All normal": no_abnorm_filter|no_history_filter, 
           "Last <2020": lastval_long_filter|no_history_filter, 
           "No history": no_history_filter, 
           "30-40": thirty_filter, 
           "40-50": fourty_filter, 
           "50-60": fifty_filter, 
           "60-70": sixty_filter}

preds_descrs={"1_lab_manualquants_month": "lab sequence", "1_clinpheno": "clinical phenotype", "2_lastval": "last value", "2_sumstats": "sumstats", "3_twosumstats": "eGFR+Cystatin C", "3_twosumstats_2": "eGFR+UACR", "3_otherlabs": "other labs", "3_registry": "registry data", "4_all": "all data", "5_icd": "ICD data", "5_atc": "ATC data"}
metrics = ["logloss", "q10", "q25", "mae"]

goal_names = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal"}
goal_names_extra = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal", "y_MEAN": "Mean"}


# Step 0&1 - Extraction and cleaning

## Kreatinine

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3020564 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=krea


In [ ]:
###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/krea_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="umol/l").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"))
)


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/krea_2025-11-05.parquet \
    --lab_name=egfr \
    --fill_missing 1 \
    --dummies 48 71 115 -1 625 \
    --abnorm_type=KDIGO-soft2 \
    --main_unit umol/l \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 2 \
    --max_z 4

## Albrumine/Creatinine in Urine

In [ ]:
# Albumine/Creatinine in Urine
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3020682 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=uacr


In [ ]:
###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/uacr_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="mg/mmol").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"), pl.len().alias("N")))


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/uacr_2025-11-05.parquet \
    --lab_name=uacr \
    --fill_missing 1 \
    --dummies -1 0.6 16.1 -1 -1 \
    --main_unit mg/mmol \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 0.01 


## Cystatin C

In [ ]:
# Cystatin C
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3030366 \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step0_extract/ \
    --lab_name=cystc


In [ ]:

###### Check abnormality medians
extract_data = pl.read_parquet("/home/ivm/valid/data/processed_data/poster/step0_extract/cystc_2025-11-05.parquet")
(extract_data
 .with_columns(pl.when(pl.col.ABNORM=="AA").then(pl.lit("HH")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .with_columns(pl.when(pl.col.ABNORM=="A").then(pl.lit("H")).otherwise(pl.col.ABNORM).alias("ABNORM"))
 .filter(~pl.col.VALUE.is_null(), pl.col.UNIT=="mg/l").group_by("ABNORM").agg(pl.col.VALUE.mean().alias("MEAN"), pl.col.VALUE.median().alias("MEDIAN"))
)


In [ ]:

###### Cleaning
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/poster/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/poster/step0_extract/cystc_2025-11-05.parquet \
    --lab_name=cystc \
    --fill_missing 1 \
    --dummies 0.61 0.89 1.79 -1 -1 \
    --abnorm_type=KDIGO-strict \
    --main_unit mg/l \
    --plot 1 \
    --keep_last_of_day 1 \
    --ref_min 2 \
    --transform_egfr 1

# Step 2 - Other diagnosis data

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                      Step 2 - Other diagnosis data              #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! python3 /home/ivm/valid/scripts/steps/step2_diags.py \
                --lab_name=egfr \
                --res_dir=/home/ivm/valid/data/processed_data/poster/step2_diags/  \
                --diag_regex="(^N18)|(^N19)|(^Z905)|(^Q600)" --med_regex="(^A10BK)"\
                --diag_excl_regex="" \
                --med_excl_regex="" \
                --fg_ver="R13"


# Step 3 - Exclusions

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                      Step 3 - Exclusions                        #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
# diag_regex = "(^KAD00)|(^KAD10)|(^KAD01)|(^KAC11)|(^KAC00)|(^KAC01)"
# table = f"`finngen-production-library.sandbox_tools_r13.finngen_r13_service_sector_detailed_longitudinal_v1`"
# query = f"""SELECT FINNGENID, SOURCE, EVENT_AGE, APPROX_EVENT_DAY, CODE1, CODE2
#                              FROM {table}
#                              WHERE (SOURCE IN ("OPER_IN", "OPER_OUT") AND REGEXP_CONTAINS(CODE1, "{diag_regex}")) OR
#                                    (SOURCE IN ("INPAT", "OUTPAT", "PRIM_OUT") AND REGEXP_CONTAINS(CODE2, "^Z905")) OR
#                                    (SOURCE IN ("INPAT", "OUTPAT", "PRIM_OUT") AND REGEXP_CONTAINS(CODE1, "^Z905"))

#                                    """
# nephro = query_to_df(query)

# # CODE1 is diagnosis and CODE2 symptom
# nephro =nephro.unpivot(index=["FINNGENID", "EVENT_AGE", "APPROX_EVENT_DAY", "SOURCE"])
# nephro = nephro.drop(["variable"]).unique()
# nephro = nephro.rename({"value":"CODE"})
# # Other codes are numbers
# nephro = nephro.filter(pl.col("CODE").is_not_null())
# nephro.write_parquet("/home/ivm/valid/data/extra_data/processed_data/step0_extract/nephro_opercodes.parquet")
# #diags = diags.filter(pl.col("CODE").str.contains("^[A-Z][0-9]"))
# print_count(nephro)
# display(nephro)

nephro = pl.read_parquet("/home/ivm/valid/data/extra_data/processed_data/step0_extract/nephro_opercodes.parquet")


In [ ]:
########### ############# ############## ############## ###########
diags_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step2_diags/egfr_R13_2025-11-05_diags.parquet")
meds_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step2_diags/egfr_R13_2025-11-05_meds.parquet")

egfr_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/egfr_d1_KDIGO-soft2_ld_2025-11-05.parquet")
cystc_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/cystc_d1_KDIGO-strict_ld_2025-11-05.parquet")
uacr_data = pl.read_parquet("/home/ivm/valid/data/processed_data/preds_2026-01/step1_clean/uacr_d1_ld_2025-11-05.parquet")


In [ ]:
########### ############# ############## ############## ###########
from diag_utils import get_abnorm_start_dates, get_data_diags

egfr_diag = get_data_diags(get_abnorm_start_dates(egfr_data.filter(pl.col.ABNORM_CUSTOM!=0.5)), 90)


## Numbers

In [ ]:

# Abnormal eGFR
########### ############# ############## ############## ###########

print("Total people")
print_count(egfr_data)
print("Total historical data")
print_count(egfr_data.filter(pl.col.DATE<base_date))
print("History of abnormal eGFR")
print_count(egfr_data
 .filter(pl.col.DATE<base_date)
 .filter(((pl.col.ABNORM_CUSTOM!=0).any().over("FINNGENID")))
)
print("History of two abnormal eGFR")
print_count(egfr_diag
 .filter(pl.col.DATA_DIAG_DATE<base_date)
)
print("After removing those with two abnormal eGFR")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
)


In [ ]:

# Diag ICD-10 code
########### ############# ############## ############## ###########

print("##### Removing ICD-10 codes")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .drop("DIAG_DATE", "DIAG").unique()
)
display((egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .with_columns(pl.col.DIAG.str.slice(0,3).alias("DIAG"))
)["DIAG"].value_counts().sort("count", descending=True))

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
)


In [ ]:

# Diag ATC code
########### ############# ############## ############## ###########

print("##### Removing ATC codes")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
  # diagnosis
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
  # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(pl.col.MED_DATE<base_date)
  .drop("MED_DATE", "MED").unique()
)

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
    # diagnosis
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
)


In [ ]:

# Nephrectomy
########### ############# ############## ############## ###########

print("##### Removing nephrectomy codes")
print_count((egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
    .filter(pl.col.APPROX_EVENT_DAY<base_date)
))

print("After removing")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
    # SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
)

In [ ]:

# Cystatin C
########### ############# ############## ############## ###########

print("##### Removing Abnormal cystatin C eGFR")
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")        
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C cases
  .filter(pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
)

print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
)


In [ ]:

# UACR
########### ############# ############## ############## ###########

print("##### Removing Abnormal urine albumin in urine ratio")
# uacr_diag = (uacr_data
#                 .filter(pl.col.DATE<base_date,pl.col.ABNORM_CUSTOM!=0)
#                 .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
# )
print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # UACR
  .filter(pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
)

print_count(egfr_data
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")    
  # Diagnosis codes
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
  # Nephro cases
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # Cystatin C
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # UACR
  .filter(~pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
)
# Interestingly the difference here is only 10% left of people with single abnormal UACR -> again back most people with only A10BK


## Excluding

In [ ]:

# Excluding
########### ############# ############## ############## ###########

base_file_name = "egfr_d1_KDIGO-soft2_ld_2025-11-05"
(egfr_data
   # No eGFR based diagnosis
  .join(egfr_diag.select("FINNGENID", "DATA_DIAG_DATE").unique(), on="FINNGENID", how="left")
  .filter(~(pl.col.DATA_DIAG_DATE<base_date)|pl.col.DATA_DIAG_DATE.is_null())
  .drop("DATA_DIAG_DATE")   
   # No official ICD-code CKD based diagnosis
  .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
  .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
  .drop("DIAG_DATE", "DIAG").unique()
   # no use of ACE, ARB, or SGLT2 use
  .join(meds_data.filter(pl.col.MED.str.starts_with("A10BK")).select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
  .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
  .drop("MED_DATE", "MED").unique()
   # No nephrectomy
  .join(nephro.select("FINNGENID", "APPROX_EVENT_DAY"), on="FINNGENID", how="left")
  .filter(~(pl.col.APPROX_EVENT_DAY<base_date)|(pl.col.APPROX_EVENT_DAY.is_null()))
  .drop("APPROX_EVENT_DAY").unique()
  # No abnormal cystatin C eGFR
  .filter(~pl.col.FINNGENID.is_in(cystc_data.filter(pl.col.DATE<base_date, pl.col.VALUE<60)["FINNGENID"]))
  # No abnormal UACR
  .filter(~pl.col.FINNGENID.is_in(uacr_data.filter(pl.col.DATE<base_date, pl.col.VALUE>=30)["FINNGENID"]))
).write_parquet(base_path+"/step2_diags/"+base_file_name+"_filtered_"+get_date()+".parquet")


# Step 3 - Labels

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Step 3 - Labels                       #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! python3 /home/ivm/valid/scripts/steps/step3_labeling.py \
    --data_path_full "$base_path"/step2_diags/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2".parquet \
    --res_dir "$base_path"/step3_labels/ \
    --lab_name "$lab_name" \
    --start_pred_date "$start_pred_date" --end_pred_date "$end_pred_date" \
    --min_age "$min_age" --max_age "$max_age" \
    --months_buffer "$months_buffer" \
    --abnorm_type "$abnorm_type" \
    --valid_pct "$`valid_pct`" \
    --test_pct "$test_pct" \
    --version "$version"


# Step 4 - Extra Data

## eGFR Sumstats

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#                           Step 4 - Extra Data                   #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
# eGFR sumstats
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --mean_impute 0

In [ ]:

########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --mean_impute 0 \
    --interpolate 1


## Cyst C Sumstats

In [ ]:
# Cyst C sumstats
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
  --res_dir "$base_path"/step4_data/ \
  --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
  --file_path_data "$base_path"/step1_clean/"$lab_name_two"_"$extra_two"_"$date_two".parquet \
  --file_path "$base_path"/step3_labels/ \
  --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"\
  --lab_name "$lab_name" \
  --start_date "$base_date"


## Other labs

In [ ]:

# Other labs
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_labs.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_lab /home/ivm/valid/data/extra_data/processed_data/step1_clean/R13_kanta_lab_min1pct-in18-70-470958totalindvs_2025-12-18.parquet \
    --dir_path_labels "$base_path"/step3_labels/ \
    --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --clean 1


In [ ]:

########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_labs.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_lab /home/ivm/valid/data/extra_data/processed_data/step1_clean/R13_kanta_lab_min1pct-in18-70-470958totalindvs_2025-12-18.parquet \
    --dir_path_labels "$base_path"/step3_labels/ \
    --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --clean 0


## ICD

In [ ]:
# ICD
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/icds_r13_2025-06-06_min1p0pct_2025-06-06.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --lab_name "$lab_name" \
            --start_date "$base_date" \
            --col_name ICD_THREE \
            --time 0 \
            --bin_count 1 \
            --months_before 0 \
            --start_year 0 \
            --min_pct 1


## ATC

In [ ]:

# ATC
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/atcs_r13_2025-06-12_min1p0pct_2025-06-12.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --lab_name "$lab_name" \
            --start_date "$base_date" \
            --col_name ATC_FIVE \
            --time 0 \
            --bin_count 1 \
            --months_before 0 \
            --start_year 0 \
            --min_pct 1

# Step 5 - XGBoost

## Just age and sex

In [ ]:
# Just age and sex
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --pred_descriptor 0_base \
            --start_date "$base_date" \
            --preds EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

## eGFR last value

In [ ]:
# eGFR last value
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 1_lastval \
            --preds IDX_QUANT_100 LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

## eGFR Sumstats

In [ ]:
# eGFR Sumstats
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 2_sumstats \
            --preds SUMSTATS LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

In [ ]:
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 2_sumstats_ip \
            --preds SUMSTATS LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_interpolate_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

## Other labs

In [ ]:
# Other labs
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 3_otherlabs \
            --preds LAB_MAT_MEAN EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labs_"$date_5".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

## Registry data

In [ ]:
# Registry data
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 3_registry \
            --preds ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --file_path_icds "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_icds_1pct_bin_"$lab_name"_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_atcs_1pct_bin_"$lab_name"_"$date_5".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 200 \
            --fit_cv 1

## All

In [ ]:
# All
########### ############# ############## ############## ###########
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --pred_descriptor 4_all \
            --start_date "$base_date" \
            --preds EDU BMI SMOKE SUMSTATS S_MEAN LAST_VAL_DIFF LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labs_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_icds_1pct_bin_"$lab_name"_"$date_4".parquet \
            --file_path_atcs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_atcs_1pct_bin_"$lab_name"_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 400 \
            --fit_cv 1

### Shap

In [ ]:
#### Shap
preds_descr="4_all"
metric = "logloss"
print(base_path+"/step5_predict/"+file_descr+"/"+train_goals[metric]+"/xgb_"+metric+"_"+preds_descr+"/models/"+lab_name+"/")
file_path = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goals[metric]+"/xgb_"+metric+"_"+preds_descr+"/models/"+lab_name+"/")
date = file_path.split(".")[0].split("preds_")[1]
file_path = base_path+"/step5_predict/"+file_descr+"/"+goal+"/xgb_"+metric+"_"+preds_descr+"/models/"+lab_name+"/"+date+"/"
      
X_all = pl.read_parquet(file_path + "Xall_unscaled_"+date+".parquet").drop("FINNGENID")
col_names = X_all.columns
scaler = pickle.load(open(file_path+"scaler_"+date+".pkl", "rb"))
X_all = scaler.transform(X_all)

model = pickle.load(open(file_path+"model_"+date+".pkl", "rb"))
explainer = shap.TreeExplainer(model)
explanation = explainer(X_all)
shap.summary_plot(explanation, feature_names=get_plot_names(col_names, "eGFR", "Cystatin C"), max_display=20)


# Step 6 - Final Fit

In [ ]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#             Step 6 - Final Fit                                  #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --pred_descriptor 4_all \
            --start_date "$base_date" \
            --preds SUMSTATS S_MEAN LAST_VAL_DIFF BMI SMOKE EDU LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_noimpute_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labs_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_icds_1pct_bin_"$lab_name"_"$date_4".parquet \
            --file_path_atcs "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_atcs_1pct_bin_"$lab_name"_"$date_4".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/eGFR_PRS.txt.gz \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --model_fit_date 2026-01-07 \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 0 \
            --n_trials 0  \
            --final_fit 1 

# Step 7 - Results compilation

In [ ]:

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#              Step 7 - Results compilation                       #
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
from model_eval_utils import bootstrap_metric, bootstrap_difference, continuous_nri, bootstrap_nri
import sklearn.metrics as skm
import numpy as np
import os.path
from delong_utils import delong_roc_test

preds_descrs={"1_clinpheno": "clinical phenotype", 
              "2_lastval": "last value", 
              "2_sumstats": "sumstats", 
              "3_otherlabs": "other labs", 
              "3_registry": "registry data", 
              "4_all": "all data"}
metrics = ["logloss", "q10", "mae"]

goal_names = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal"}
goal_names_extra = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal", "y_MEAN": "Mean"}
history_filters = {"All": True, 
           "History": history_filter, 
           "All normal | No history": no_abnorm_filter|no_history_filter, 
           "Last <2020 | No history": lastval_long_filter|no_history_filter, 
           "Last <2020": lastval_long_filter, 
           "No history": no_history_filter}
filters = {"All": True, 
           "History": history_filter, 
           "All normal | No history": no_abnorm_filter|no_history_filter, 
           "Last <2020 | No history": lastval_long_filter|no_history_filter, 
           "Last <2020": lastval_long_filter, 
           "No history": no_history_filter, 
           "30-40": thirty_filter, 
           "40-50": fourty_filter, 
           "50-60": fifty_filter, 
           "60-70": sixty_filter}

set_names = {0: "Train", 1: "Valid", 2: "Test", 10: "TrainValid"}


In [ ]:

########### ############# ############## ############## ###########
no_dups_combos = [
    ("xgb_logloss_0_base", "xgb_logloss_2_sumstats"),
    ("xgb_logloss_0_base", "xgb_logloss_3_registry"),
    ("xgb_logloss_0_base", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_0_base", "xgb_logloss_4_all"),

    ("xgb_logloss_2_sumstats", "xgb_logloss_3_registry"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_4_all"),
    
    ("xgb_logloss_3_registry", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_3_registry", "xgb_logloss_4_all"),
    ("xgb_logloss_3_otherlabs", "xgb_logloss_4_all"),
]    

## Differences

In [ ]:

# Differences
########### ############# ############## ############## ###########
###### metric = "logloss"
results = pl.DataFrame()
#results = pl.read_csv("/home/ivm/valid/results/model_evals/egfr/egfr_testv10_2022_w3_diffs_2025-10-20.csv")
for combo_1, combo_2 in no_dups_combos:
    for train_goal_1 in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
        for train_goal_2 in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
            if combo_1 != "maxim":
                file_path_1 = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goal_1+"/"+combo_1+"/models/"+lab_name+"/")
                metric_1 = combo_1.split("_")[1]
            else:
                file_path_1 = maxim_file_path
                metric_1 = "focal"
            if combo_2 not in ["maxim", "maxim_2"]:
                file_path_2 = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goal_2+"/"+combo_2+"/models/"+lab_name+"/")
                metric_2 = combo_2.split("_")[1]
            else:
                metric_2 = "focal"
                if combo_2 == "maxim": file_path_2 = maxim_file_path
                elif combo_2 == "maxim_2": file_path_2 = maxim_2_file_path
                # Maxim was trained on y_MEAN
                if train_goal_2 == "y_NEXT_ABNORM": continue
            if not file_path_1 or not file_path_2 or file_path_1==file_path_2: continue
            preds_1 = pl.read_parquet(file_path_1)
            preds_2 = pl.read_parquet(file_path_2)
            if metric_1 == "logloss" and metric_2 in ["logloss", "focal"]:
                for set_no in set_names:
                    descriptors = {"MODEL_1": combo_1, "TRAIN_1": train_goal_1, "MODEL_2": combo_2, "TRAIN_2": train_goal_2, "SET": set_names[set_no]}
                    for crnt_filter_name in filters:
                        descriptors["FILTER"] = crnt_filter_name
        
                        for goal_name in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
                            descriptors["EVAL"]=goal_name
        
                            preds = preds_1.select("FINNGENID", "SET", "EVENT_AGE", goal_name, "ABNORM_PROBS").join(preds_2.select("FINNGENID", "ABNORM_PROBS"), on="FINNGENID", how="left")
                            preds = preds.filter(~pl.col.ABNORM_PROBS.is_null(), ~pl.col.ABNORM_PROBS_right.is_null())
                            if set_no==10:
                                crnt_preds = preds.filter(pl.col.SET!=2).filter(filters[crnt_filter_name]) 
                            else:
                                crnt_preds = preds.filter(pl.col.SET==set_no).filter(filters[crnt_filter_name])
                            if (crnt_preds[goal_name]==1).sum()<10: 
                                print("skipping: "+crnt_filter_name)
                                continue

                            ### P-values for AUCs with DeLong
                            pval_diff = 10**delong_roc_test(crnt_preds[goal_name].to_numpy(), crnt_preds["ABNORM_PROBS"].to_numpy(), crnt_preds["ABNORM_PROBS_right"].to_numpy())[0]
        
                            diff_est, lowci, highci, _, avg_1, avg_2 = bootstrap_difference(metric_func = (skm.roc_auc_score),
                                                                                                    preds_1=crnt_preds["ABNORM_PROBS"].to_numpy(), 
                                                                                                    preds_2=crnt_preds["ABNORM_PROBS_right"].to_numpy(),
                                                                                                    obs=crnt_preds[goal_name].to_numpy(),
                                                                                                    n_boots=100)
                            descriptors["AUCDiff_Pvalue"]=pval_diff
                            descriptors["AUCDiff_Est"]=diff_est
                            descriptors["AUCDiff_CIneg"]=lowci
                            descriptors["AUCDiff_CIpos"]=highci
                            ### P-values for Average Precision with Bootstrapping
                            diff_est, lowci, highci, pval_diff, avg_1, avg_2 = bootstrap_difference(metric_func = (skm.average_precision_score),
                                                                                                    preds_1=crnt_preds["ABNORM_PROBS"].to_numpy(), 
                                                                                                    preds_2=crnt_preds["ABNORM_PROBS_right"].to_numpy(),
                                                                                                    obs=crnt_preds[goal_name].to_numpy(),
                                                                                                    n_boots=100)
                            descriptors["AvgPrecDiff_Pvalue"]=pval_diff
                            descriptors["AvgPrecDiff_Est"]=diff_est
                            descriptors["AvgPrecDiff_CIneg"]=lowci
                            descriptors["AvgPrecDiff_CIpos"]=highci
        
                            ### NRI with CI measure of if new model is better at reclassification <0 -> worst and >0 -> better
                            nri, lowci, highci = bootstrap_nri(continuous_nri, 
                                                               crnt_preds[goal_name].to_numpy(), 
                                                               crnt_preds["ABNORM_PROBS"].to_numpy(),
                                                               crnt_preds["ABNORM_PROBS_right"].to_numpy(),
                                                               n_boots=100)
                            descriptors["NRI"]=nri
                            descriptors["NRI_CI"]="("+str(round(lowci, 2))+ "-"+ str(round(highci, 2)) + ")"
        
                            results = pl.concat([results, pl.DataFrame(descriptors)])
            display(results.filter(pl.col.FILTER=="All", pl.col.SET=="Test", pl.col.EVAL=="y_NEXT_ABNORM"))
make_dir(base_path+"/model_evals/"+lab_name+"/")
results.write_csv(base_path+"/model_evals/"+lab_name+"_"+file_descr+"_diffs_"+get_date()+".csv")

## Aucs

In [ ]:

# AUCs
########### ############# ############## ############## ###########
results = pl.DataFrame()
labels = pl.read_parquet(base_path+"/step3_labels/egfr_d1_KDIGO-soft2_ld_2025-11-05_filtered_2026-01-07_testv10_2022_w3_2026-01-07_labels.parquet")
for crnt_option in set([elem_1 for elem_1, elem_2 in no_dups_combos] + [elem_2 for elem_1, elem_2 in no_dups_combos]):
    ### Getting info
    if True:
        for train_goal in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
            if crnt_option != "maxim":
                metric = crnt_option.split("_")[1]
                if metric != "logloss": continue
                mdl_name = crnt_option.split("_")[0]
                crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
                # Getting data
                file_path = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goal+"/"+crnt_option+"/models/"+lab_name+"/")
                if not file_path: continue
                date = file_path.split(".")[0].split("preds_")[1]
                preds = pl.read_parquet(file_path)
            else:
                if train_goal == "y_NEXT_ABNORM": continue
                mdl_name = crnt_option
                if crnt_option == "maxim": 
                    crnt_pred_descr = "5_all_pgs"
                    file_path = maxim_file_path
                else: 
                    crnt_pred_descr = "92_sumstats_pgs"
                    file_path = maxim_2file_path
                metric = "logloss"
                preds = pl.read_parquet(file_path)
                preds = preds.join(labels.select("FINNGENID", "SET", "y_MEAN_ABNORM", "y_NEXT_ABNORM", "EVENT_AGE"), on="FINNGENID")

            
        
            mdl_name = crnt_option.split("_")[0]
            crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
            descriptors = {"Date": date, 
                           "Model": mdl_name, 
                           "Train Goal": train_goal, 
                           "File Description": file_descr, 
                           "Predictors": crnt_pred_descr,
                           "Outcome": goal_names[train_goal]}
            for crnt_filter_name in filters:
                filter_preds = preds.filter(filters[crnt_filter_name])
            
                for set_no in [0,1,2]:
                    if set_no == 10: crnt_preds=filter_preds.filter(pl.col.SET!=2)
                    else: crnt_preds=filter_preds.filter(pl.col.SET==set_no)
                    if crnt_preds.height == 0: continue
                    descriptors["FILTER"] = crnt_filter_name
                    descriptors["SET"] = set_names[set_no]
                    for goal in ["y_MEAN_ABNORM", "y_NEXT_ABNORM"]:
                        descriptors["Eval goal"] = goal
                        N_total = crnt_preds.height
                        N_cases = crnt_preds.filter(pl.col(goal)==1).height
                        if N_cases < 10: continue
                        descriptors["N_TOTAL"] = N_total
                        descriptors["N_CASE"] = N_cases

                        AUC = bootstrap_metric(skm.roc_auc_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["AUC"] = AUC[0]
                        descriptors["AUC_CIneg"] = AUC[1]
                        descriptors["AUC_CIpos"] = AUC[2]
                
                        averagePrec = bootstrap_metric(skm.average_precision_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["avgPrec"] = averagePrec[0]
                        descriptors["avgPrec_CIneg"] = averagePrec[1]
                        descriptors["avgPrec_CIpos"] = averagePrec[2]
            
                        descriptors["Brier"] = skm.brier_score_loss(crnt_preds[goal], 
                                                                    crnt_preds["ABNORM_PROBS"])
            
                
                        results = pl.concat([results, pl.DataFrame(descriptors)])
    display(results)
results.filter(pl.col.N_CASE>=5).write_csv(base_path+"/model_evals/"+lab_name+"/"+lab_name+"_"+file_descr+"_aucs_"+get_date()+".csv")


### Above 50

In [ ]:

########### ############# ############## ############## ###########

results = pl.DataFrame()

for crnt_option in set([elem_1 for elem_1, elem_2 in no_dups_combos] + [elem_2 for elem_1, elem_2 in no_dups_combos]):
    ### Getting info
    metric = crnt_option.split("_")[1]
    if metric != "logloss": continue
    if True:
        for train_goal in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
            mdl_name = crnt_option.split("_")[0]
            crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
            # Getting data
            file_path = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goal+"/"+crnt_option+"/models/"+lab_name+"/")
            if not file_path: continue
            date = file_path.split(".")[0].split("preds_")[1]
            preds = pl.read_parquet(file_path)
        
            mdl_name = crnt_option.split("_")[0]
            crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
            descriptors = {"Date": date, 
                           "Model": mdl_name, 
                           "Train Goal": train_goal, 
                           "File Description": file_descr, 
                           "Predictors": crnt_pred_descr,
                           "Outcome": goal_names[train_goal]}
            for crnt_filter_name in history_filters:
                filter_preds = preds.filter(history_filters[crnt_filter_name]).filter(pl.col.EVENT_AGE>=50)
            
                for set_no in [0,1,2,10]:
                    if set_no == 10: crnt_preds=filter_preds.filter(pl.col.SET!=2)
                    else: crnt_preds=filter_preds.filter(pl.col.SET==set_no)
                    if crnt_preds.height == 0: continue
                    descriptors["FILTER"] = crnt_filter_name
                    descriptors["SET"] = set_names[set_no]
                    for goal in ["y_MEAN_ABNORM", "y_NEXT_ABNORM"]:
                        descriptors["Eval goal"] = goal
                        N_total = crnt_preds.height
                        N_cases = crnt_preds.filter(pl.col(goal)==1).height
                        descriptors["N_TOTAL"] = N_total
                        descriptors["N_CASE"] = N_cases
                        if N_cases <5: continue

                        AUC = bootstrap_metric(skm.roc_auc_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["AUC"] = AUC[0]
                        descriptors["AUC_CIneg"] = AUC[1]
                        descriptors["AUC_CIpos"] = AUC[2]
                
                        averagePrec = bootstrap_metric(skm.average_precision_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["avgPrec"] = averagePrec[0]
                        descriptors["avgPrec_CIneg"] = averagePrec[1]
                        descriptors["avgPrec_CIpos"] = averagePrec[2]
            
                        descriptors["Brier"] = skm.brier_score_loss(crnt_preds[goal], 
                                                                    crnt_preds["ABNORM_PROBS"])
            
                
                        results = pl.concat([results, pl.DataFrame(descriptors)])
    display(results)
results.filter(pl.col.N_CASE>=5).write_csv(base_path+"/model_evals/"+lab_name+"/"+lab_name+"_"+file_descr+"_aucs_gg50_"+get_date()+".csv")


### Below 50

In [ ]:

########### ############# ############## ############## ###########

results = pl.DataFrame()

for crnt_option in set([elem_1 for elem_1, elem_2 in no_dups_combos] + [elem_2 for elem_1, elem_2 in no_dups_combos]):
    ### Getting info
    metric = crnt_option.split("_")[1]
    if metric != "logloss": continue
    if True:
        for train_goal in ["y_NEXT_ABNORM", "y_MEAN_ABNORM"]:
            mdl_name = crnt_option.split("_")[0]
            crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
            # Getting data
            file_path = get_dated_path(base_path+"/step5_predict/"+file_descr+"/"+train_goal+"/"+crnt_option+"/models/"+lab_name+"/")
            if not file_path: continue
            date = file_path.split(".")[0].split("preds_")[1]
            preds = pl.read_parquet(file_path)
        
            mdl_name = crnt_option.split("_")[0]
            crnt_pred_descr = "_".join([elem for elem in crnt_option.split("_") if elem not in ["lr", "xgb", "GRU", "logloss", "TLSTM", "mae", "q75", "q10", "q25"]])
            descriptors = {"Date": date, 
                           "Model": mdl_name, 
                           "Train Goal": train_goal, 
                           "File Description": file_descr, 
                           "Predictors": crnt_pred_descr,
                           "Outcome": goal_names[train_goal]}
            for crnt_filter_name in history_filters:
                filter_preds = preds.filter(history_filters[crnt_filter_name]).filter(pl.col.EVENT_AGE<50)
            
                for set_no in [0,1,2,10]:
                    if set_no == 10: crnt_preds=filter_preds.filter(pl.col.SET!=2)
                    else: crnt_preds=filter_preds.filter(pl.col.SET==set_no)
                    if crnt_preds.height == 0: continue
                    descriptors["FILTER"] = crnt_filter_name
                    descriptors["SET"] = set_names[set_no]
                    for goal in ["y_MEAN_ABNORM", "y_NEXT_ABNORM"]:
                        descriptors["Eval goal"] = goal
                        N_total = crnt_preds.height
                        N_cases = crnt_preds.filter(pl.col(goal)==1).height
                        descriptors["N_TOTAL"] = N_total
                        descriptors["N_CASE"] = N_cases
                        if N_cases <5: continue

                        AUC = bootstrap_metric(skm.roc_auc_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["AUC"] = AUC[0]
                        descriptors["AUC_CIneg"] = AUC[1]
                        descriptors["AUC_CIpos"] = AUC[2]
                
                        averagePrec = bootstrap_metric(skm.average_precision_score, 
                                               crnt_preds[goal],
                                               crnt_preds["ABNORM_PROBS"],
                                               n_boots=100)
                        descriptors["avgPrec"] = averagePrec[0]
                        descriptors["avgPrec_CIneg"] = averagePrec[1]
                        descriptors["avgPrec_CIpos"] = averagePrec[2]
            
                        descriptors["Brier"] = skm.brier_score_loss(crnt_preds[goal], 
                                                                    crnt_preds["ABNORM_PROBS"])
            
                
                        results = pl.concat([results, pl.DataFrame(descriptors)])
    display(results)
results.filter(pl.col.N_CASE>=5).write_csv(base_path+"/model_evals/"+lab_name+"/"+lab_name+"_"+file_descr+"_aucs_b50_"+get_date()+".csv")
